# Tái hiện ICAI 2026 - Acacia Hòa Vang (Kaggle, chạy tự động)

**Cách dùng:** bấm **Save Version -> Save & Run All (Commit)**. Notebook tự làm lần lượt mọi thí nghiệm (47 lần huấn luyện),
bỏ qua lần đã xong, tự dừng an toàn trước giới hạn 12 giờ của Kaggle (để kết quả được lưu), và khi đã đủ thì tự phân tích + xuất báo cáo.

**Lần sau (tiếp tục):** khung Input bên phải -> mục Notebook Output của chính notebook này -> chọn **version mới nhất**
(lần đầu: Add Input -> Your Work -> chọn notebook này). Rồi lại Save & Run All. Lặp đến khi cell cuối báo **HOÀN TẤT**.

Chạy thử pipeline trước (vài phút): đặt `MODE = 'smoke'` và bấm Run All ở chế độ tương tác.

In [ ]:
MODE = 'auto'          # 'auto' = chạy tiếp mọi thứ còn thiếu | 'smoke' = chạy thử nhanh
SOFT_HOURS = 10.5      # sau mốc này không bắt đầu lần huấn luyện mới
HARD_HOURS = 11.5      # mốc này mà lần đang chạy chưa xong thì dừng nó (sẽ chạy lại từ đầu ở phiên sau)

### Cell 1 - cài đặt, tự tìm dataset

In [ ]:
import os, shutil, subprocess, sys, glob, time, signal, threading
T0 = time.time()

def find_dir(marker, root='/kaggle/input', depth=6):
    """Tìm thư mục chứa `marker` (đường dẫn tương đối) trong /kaggle/input."""
    for d, subs, _ in os.walk(root):
        if d.count(os.sep) - root.count(os.sep) >= depth:
            subs[:] = []
            continue
        if '/work/' in d + '/' or d.endswith('/code'):
            continue   # bỏ qua output của phiên trước
        if os.path.exists(os.path.join(d, marker)):
            return d
    return None

CODE = find_dir('src/acacia/runner.py')
DATA = find_dir('cache/index.json')
print('CODE =', CODE); print('DATA =', DATA)
assert CODE, 'Không thấy dataset code (thư mục chứa src/acacia). Kiểm tra Input.'
assert DATA, 'Không thấy dataset data (thư mục chứa cache/index.json). Kiểm tra Input.'
assert os.path.exists(f'{DATA}/data/grid.csv'), f'Thiếu {DATA}/data/grid.csv'

W = '/kaggle/working'
shutil.copytree(CODE, f'{W}/code', dirs_exist_ok=True)
os.chdir(f'{W}/code')
os.environ['ACACIA_DATA']  = f'{DATA}/data'
os.environ['ACACIA_CACHE'] = f'{DATA}/cache'
os.environ['ACACIA_WORK']  = f'{W}/work'

def elapsed_h():
    return (time.time() - T0) / 3600

def sh(cmd, must=True, timeout=None):
    """Chạy lệnh, in log trực tiếp. Lỗi -> dừng notebook. Quá timeout (giây) -> dừng lệnh, trả về 'timeout'."""
    print(); print('$', cmd, flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         start_new_session=True)
    killed = []
    def kill():
        killed.append(1)
        try: os.killpg(p.pid, signal.SIGKILL)
        except Exception: pass
    timer = threading.Timer(timeout, kill) if timeout else None
    if timer: timer.start()
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if timer: timer.cancel()
    print(f'[xong sau {(time.time()-t)/60:.1f} phút, mã thoát {rc}]', flush=True)
    if killed:
        return 'timeout'
    if rc != 0 and must:
        raise RuntimeError(f'LỖI khi chạy: {cmd}  -> xem log phía trên')
    return rc

sh('pip install -q tabulate xgboost pyyaml')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv', must=False)
import torch; print('CUDA:', torch.cuda.is_available())

### Cell 2 - chép kết quả các phiên trước (nếu đã gắn Notebook Output)

In [ ]:
prev = sorted(glob.glob('/kaggle/input/**/work/runs', recursive=True))
for r in prev:
    src = os.path.dirname(r)
    n = len(glob.glob(f'{r}/*/*/s*/summary.json'))
    print(f'chép {src} ({n} lần huấn luyện đã xong) -> {W}/work')
    shutil.copytree(src, f'{W}/work', dirs_exist_ok=True)
if not prev:
    print('Không có kết quả phiên trước (bình thường ở lần chạy đầu).')

### Cell 3 - chuẩn bị dữ liệu (chia tập, chuẩn hóa) + kiểm tra mốc bài báo

In [ ]:
sh('python scripts/02_prepare.py')
assert os.path.exists(f'{W}/work/prepared/norm_stats.json'), 'chưa tạo được norm_stats.json'
sh('python -m pytest -q tests', must=False)   # mong đợi: 18 passed

### Cell 4 - huấn luyện tự động (xếp hàng mọi lần chạy, bỏ qua lần đã xong)

In [ ]:
import yaml
BASE_SEEDS = yaml.safe_load(open('configs/base.yaml', encoding='utf8'))['seeds']
# thứ tự ưu tiên: kết quả chính trước; modality trước context_sweep vì optical dropout cần modality/s1s2_q
ORDER = ['main_test', 'modality', 'context_sweep', 'quast_ablation', 'short_window', 'buffer', 'capacity']
if MODE == 'smoke':
    ORDER = ['smoke']
MAIN_PRIORITY = ['unet', 'quast', 'utae', 'tsvit']

def all_jobs():
    jobs = []
    for e in ORDER:
        ex = yaml.safe_load(open(f'configs/experiments/{e}.yaml', encoding='utf8'))
        runs = list(ex['runs'])
        if e == 'main_test':
            runs.sort(key=lambda r: MAIN_PRIORITY.index(r) if r in MAIN_PRIORITY else 99)
        for s in ex.get('seeds', BASE_SEEDS):          # theo seed trước -> sớm có đủ 4 mô hình
            for run in runs:
                jobs.append((e, run, s, ex.get('eval_splits', ['val'])))
    return jobs

def run_path(e, run, s):
    return f'{W}/work/runs/{e}/{run}/s{s}'

def is_done(e, run, s, splits):
    d = run_path(e, run, s)
    return os.path.exists(f'{d}/summary.json') and all(os.path.exists(f'{d}/eval_{sp}.json') for sp in splits)

JOBS = all_jobs()
todo = [j for j in JOBS if not is_done(*j)]
print(f'Tổng {len(JOBS)} lần chạy, đã xong {len(JOBS)-len(todo)}, còn {len(todo)}. Đã dùng {elapsed_h():.2f} giờ.')
STOPPED = False
for (e, run, s, splits) in todo:
    if elapsed_h() > SOFT_HOURS:
        print(f'\n>>> Đã {elapsed_h():.1f} giờ - dừng để kịp lưu kết quả. Bấm Save & Run All lần nữa để chạy tiếp.')
        STOPPED = True
        break
    d = run_path(e, run, s)
    if os.path.isdir(d) and not os.path.exists(f'{d}/summary.json'):
        print('xóa lần chạy dở dang:', d)
        shutil.rmtree(d)
    extra = '--set train.num_workers=2' if MODE == 'smoke' else '--set train.num_workers=4'
    r = sh(f'python scripts/03_run.py {e} --only {run} --seeds {s} {extra}',
           timeout=max(60, (HARD_HOURS - elapsed_h()) * 3600))
    if r == 'timeout':
        print(f'\n>>> Hết giờ giữa chừng {e}/{run}/s{s} - lần này sẽ chạy lại ở phiên sau.')
        if os.path.isdir(d) and not os.path.exists(f'{d}/summary.json'):
            shutil.rmtree(d)
        STOPPED = True
        break
left = [j for j in JOBS if not is_done(*j)]
print(f'\nCòn {len(left)}/{len(JOBS)} lần chạy chưa xong.')

### Cell 5 - tóm tắt thời gian / kết quả các lần đã huấn luyện

In [ ]:
import json, pandas as pd
rows = []
for f in sorted(glob.glob(f'{W}/work/runs/*/*/s*/summary.json')):
    s = json.load(open(f)); p = f.split('/')
    e = {}
    for sp in ('val', 'test'):
        ef = f.replace('summary.json', f'eval_{sp}.json')
        if os.path.exists(ef):
            e[f'{sp}_mIoU'] = round(json.load(open(ef))['scores']['mIoU'], 4)
    rows.append({'exp': p[-4], 'run': p[-3], 'seed': p[-2], 'epochs': s['epochs_run'], 'best_ep': s['best_epoch'],
                 'phut': round(s['train_time_s'] / 60, 1), **e})
df = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(df.to_string() if len(df) else 'chưa có lần chạy nào')
if len(df):
    print(); print('Tổng thời gian huấn luyện: %.1f giờ' % (df.phut.sum() / 60))

### Cell 6 - phân tích + báo cáo (tự chạy khi mọi lần huấn luyện đã xong)

In [ ]:
if MODE == 'auto' and not left and elapsed_h() < HARD_HOURS:
    FAILED = []
    for c in ['04_optical_dropout.py --extra', '05_stats.py', '06_polygon.py', '07_screening.py', '08_report.py']:
        if sh(f'python scripts/{c}', must=False) != 0:     # lỗi một bước phân tích không làm mất output
            FAILED.append(c)
    rep = f'{W}/work/results/report/REPORT.md'
    if os.path.exists(rep):
        from IPython.display import Markdown, display
        display(Markdown(open(rep, encoding='utf8').read()))
    print()
    if FAILED:
        print('>>> Các bước phân tích bị lỗi (xem log phía trên):', FAILED)
    else:
        print('================ HOÀN TẤT: tải results.zip ở tab Output ================')
elif MODE == 'smoke':
    print("Smoke xong - đổi MODE = 'auto' rồi Save & Run All (Commit) để chạy thật.")
else:
    print(f'CHƯA XONG: còn {len(left)} lần huấn luyện. Cập nhật Input lên version mới nhất của notebook này rồi Save & Run All lần nữa.')

### Cell 7 - nén bảng/hình để tải về (checkpoint vẫn nằm trong Output để phiên sau dùng tiếp)

In [ ]:
sh(f'cd {W} && rm -f results.zip && zip -qr results.zip work/results work/prepared work/runs -x "*.pt" "*/pred_*.npz" && ls -lh results.zip', must=False)